# O-RAG Quality Evaluation
This notebook evaluates the RAG system's performance using an LLM-as-a-judge approach via OpenRouter.

In [ ]:
import os
import sys
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Add the RAG project path to sys.path
project_root = r'd:\orag'
python_scripts_path = os.path.join(project_root, 'orag', 'android', 'app', 'src', 'main', 'python')
if python_scripts_path not in sys.path:
    sys.path.append(python_scripts_path)

print(f"Project root: {project_root}")
print(f"Python scripts path: {python_scripts_path}")

## 1. Initialize RAG Components
We need to initialize the database and the retriever. We'll also ingest a sample document if none exists.

In [ ]:
import pipeline
from storage import init_db
from downloader import set_model_dir

# Set model directory (adjust if models are stored elsewhere)
model_dir = os.path.join(project_root, 'models')
if not os.path.exists(model_dir):
    os.makedirs(model_dir, exist_ok=True)
set_model_dir(model_dir)

# Initialize DB and Pipeline
init_db()
pipeline.init(model_dir)

print("RAG System Initialized.")

## 2. Ingest Document
We'll ingest `Learning_Python.pdf` for our evaluation.

In [ ]:
pdf_path = os.path.join(project_root, 'Learning_Python.pdf')
if os.path.exists(pdf_path):
    print(f"Ingesting {pdf_path}...")
    success, message = pipeline.ingest_document(pdf_path)
    print(f"Ingestion Result: {success}, {message}")
else:
    print(f"Warning: {pdf_path} not found. Please ensure the file exists.")

## 3. Define Evaluation Questions
We'll define a set of questions to test the RAG system.

In [ ]:
eval_questions = [
    "What are the main advantages of using Python?",
    "Explain the concept of list comprehensions in Python.",
    "How does Python handle memory management?",
    "What is the difference between a list and a tuple?",
    "How do you define a class in Python?",
    "What is the purpose of the '__init__' method?",
    "Explain the 'self' parameter in class methods.",
    "How does inheritance work in Python?",
    "What are decorators and how are they used?",
    "Describe the Global Interpreter Lock (GIL) and its impact."
]

results = []

## 4. Run RAG and Collect Responses
We'll iterate through the questions and store the answers and retrieved contexts.

In [ ]:
print("Running RAG queries...")
for q in tqdm(eval_questions):
    success, answer, sources = pipeline.ask(q)
    if success:
        results.append({
            "question": q,
            "answer": answer,
            "contexts": [s['chunk_text'] for s in sources]
        })
    else:
        print(f"Failed to get answer for: {q}")

print(f"Collected {len(results)} responses.")

## 5. LLM-as-a-Judge Evaluation (OpenRouter)
We'll use OpenRouter to evaluate the quality of the responses.

In [ ]:
import requests

OPENROUTER_API_KEY = "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"
JUDGE_MODEL = "google/gemini-2.0-flash-001" # or "anthropic/claude-3.5-sonnet"

def call_openrouter(prompt):
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        data=json.dumps({
            "model": JUDGE_MODEL,
            "messages": [
                {"role": "system", "content": "You are an expert RAG evaluator. Provide scores between 0 and 1 for the given metrics. Output ONLY a JSON object."},
                {"role": "user", "content": prompt}
            ]
        })
    )
    return response.json()['choices'][0]['message']['content']

def evaluate_response(item):
    prompt = f"""
    Evaluate the following RAG response based on these metrics:
    1. Faithfulness: Is the answer derived solely from the provided contexts? (0-1)
    2. Answer Relevance: Does the answer address the question effectively? (0-1)
    3. Context Precision: Are the retrieved contexts relevant to answering the question? (0-1)

    Question: {item['question']}
    Contexts: {json.dumps(item['contexts'])}
    Answer: {item['answer']}

    Output format (JSON only):
    {{
        "faithfulness": score,
        "answer_relevance": score,
        "context_precision": score,
        "reasoning": "brief explanation"
    }}
    """
    try:
        response_text = call_openrouter(prompt)
        # Clean response text if LLM adds markdown triple backticks
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
             response_text = response_text.split("```")[1].split("```")[0].strip()
             
        return json.loads(response_text)
    except Exception as e:
        print(f"Error evaluating: {e}")
        return {"faithfulness": 0, "answer_relevance": 0, "context_precision": 0, "reasoning": str(e)}

print("Evaluating responses with LLM-as-a-judge...")
eval_results = []
for res in tqdm(results):
    scores = evaluate_response(res)
    eval_results.append({**res, **scores})

df_eval = pd.DataFrame(eval_results)
df_eval.to_csv("rag_eval_results.csv", index=False)
print("Evaluation complete.")

## 6. Visualizations
We'll visualize the evaluation scores.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Average Scores
avg_scores = df_eval[['faithfulness', 'answer_relevance', 'context_precision']].mean()
avg_scores.plot(kind='bar', color=['#4C72B0', '#55A868', '#C44E52'])
plt.title("Average RAG Quality Scores")
plt.ylabel("Score (0-1)")
plt.ylim(0, 1.1)
plt.xticks(rotation=45)
plt.show()

# Distribution of scores
plt.figure(figsize=(12, 5))
df_melted = df_eval.melt(id_vars=['question'], value_vars=['faithfulness', 'answer_relevance', 'context_precision'], 
                        var_name='Metric', value_name='Score')
sns.boxplot(x='Metric', y='Score', data=df_melted)
plt.title("Score Distribution across Metrics")
plt.ylim(0, 1.1)
plt.show()

In [ ]:
print("--- Detailed Results Summary ---")
print(df_eval[['question', 'faithfulness', 'answer_relevance', 'context_precision']].to_string())